# 🤖 3주차 실습 | AI 기초 강의
## 머신러닝의 작동 원리와 성능 평가

---

### 📌 실습 목표
| 순서 | 내용 |
|------|------|
| 1 | Feature와 Label 이해하기 |
| 2 | 실제값(actual)과 예측값(pred) 비교 |
| 3 | TP / TN / FP / FN 직접 계산하기 |
| 4 | Accuracy / Precision / Recall / F1 계산 |
| 5 | sklearn 라이브러리로 한 번에 확인하기 |
| 6 | 나만의 데이터로 바꿔보기 (응용) |

---
### ▶ 실습 방법
- 각 셀 왼쪽의 **▶ 버튼**을 클릭하면 코드가 실행됩니다.
- 위에서 아래 순서대로 실행하세요.
- `# TODO` 표시된 곳은 여러분이 직접 채워야 합니다!


---
## 📦 STEP 0. 준비 — 필요한 라이브러리 불러오기
아래 셀을 먼저 실행해주세요. (한 번만 실행하면 됩니다)

In [ ]:
# 기본 라이브러리
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# 머신러닝 평가 지표
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print('✅ 준비 완료! 이제 아래 셀부터 순서대로 실행하세요.')

---
## 📊 STEP 1. Feature와 Label 이해하기

머신러닝에서 가장 중요한 두 가지 개념입니다.

- **Feature**: AI가 보고 판단하는 입력 정보 (예: 공부시간, 출석률)
- **Label**: AI가 맞혀야 하는 정답 (예: 합격/불합격)

In [ ]:
# ─── 학생 데이터 만들기 ────────────────────────────────────────────
data = {
    '공부시간(h)': [2, 5, 4, 1, 6, 3, 7, 2, 5, 1],
    '출석률(%)':   [60, 80, 75, 50, 90, 65, 95, 55, 85, 45],
    '과제제출(개)': [3, 5, 4, 2, 6, 3, 7, 2, 5, 1],
    '합격여부':    [0, 1, 1, 0, 1, 0, 1, 0, 1, 0]  # 0=불합격, 1=합격
}

df = pd.DataFrame(data)

# 표시 설정
def highlight_label(col):
    if col.name == '합격여부':
        return ['background-color: #d0f0e4; color: #1a7a72; font-weight: bold'] * len(col)
    return ['background-color: #dceeff'] * len(col)

print('📋 학생 데이터 (파란색 = Feature, 초록색 = Label)')
df.style.apply(highlight_label)

In [ ]:
# ─── Feature와 Label 분리 ─────────────────────────────────────────
features = df[['공부시간(h)', '출석률(%)', '과제제출(개)']]
labels   = df['합격여부']

print('📐 Feature (입력 정보):')
print(features)
print()
print('🎯 Label (정답):')
print(labels.to_frame())

In [ ]:
# ─── 시각화: 공부시간 vs 합격여부 ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ['#E05C5C' if y == 0 else '#4CAF82' for y in labels]

# 공부시간 vs 합격
axes[0].scatter(df['공부시간(h)'], df['출석률(%)'],
                c=colors, s=120, edgecolors='white', linewidth=1.5)
axes[0].set_xlabel('공부시간 (h)', fontsize=12)
axes[0].set_ylabel('출석률 (%)', fontsize=12)
axes[0].set_title('Feature 시각화 (빨강=불합격, 초록=합격)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# 합격/불합격 비율
counts = labels.value_counts()
axes[1].bar(['불합격(0)', '합격(1)'], counts.values,
            color=['#E05C5C', '#4CAF82'], edgecolor='white')
axes[1].set_ylabel('학생 수', fontsize=12)
axes[1].set_title('Label 분포', fontsize=12)
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 0.05, str(v), ha='center', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print('💡 공부시간이 길고 출석률이 높을수록 초록색(합격)이 많습니다!')

---
## 🔍 STEP 2. 실제값(actual)과 예측값(pred) 비교

머신러닝 모델이 예측한 결과를 실제 정답과 비교합니다.

In [ ]:
# ─── 실제값과 AI 예측값 ────────────────────────────────────────────
#  실제 정답 (이미 알고 있는 값)
actual = [1, 0, 1, 0, 1, 1, 0, 0, 1, 0]

# AI 모델이 예측한 값
pred   = [1, 1, 1, 0, 0, 1, 0, 0, 1, 0]

print('=' * 50)
print(f'실제값(actual): {actual}')
print(f'예측값(pred  ): {pred}')
print('=' * 50)

# 맞춘 것과 틀린 것 표시
print('\n결과 비교 (O=맞음, X=틀림):')
result = []
for i, (a, p) in enumerate(zip(actual, pred)):
    mark = '✅' if a == p else '❌'
    result.append(f'{i+1}번: 실제={a} 예측={p} {mark}')

for r in result:
    print(r)

---
## 🔢 STEP 3. TP / TN / FP / FN 직접 계산하기

| 용어 | 뜻 | 예시 |
|------|------|------|
| **TP** (True Positive) | 실제 양성 → 양성으로 예측 ✅ | 환자를 환자라고 맞게 탐지 |
| **TN** (True Negative) | 실제 음성 → 음성으로 예측 ✅ | 건강한 사람을 건강하다고 맞게 판단 |
| **FP** (False Positive) | 실제 음성 → 양성으로 예측 ⚠️ | 건강한데 환자라고 잘못 판단 (오경보) |
| **FN** (False Negative) | 실제 양성 → 음성으로 예측 🚨 | 환자를 건강하다고 놓침 (위험!) |

In [ ]:
# ─── 방법 1: 직접 for문으로 계산 ────────────────────────────────────
tp, tn, fp, fn = 0, 0, 0, 0

for a, p in zip(actual, pred):
    if   a == 1 and p == 1:  tp += 1   # 실제 양성, 양성 예측
    elif a == 0 and p == 0:  tn += 1   # 실제 음성, 음성 예측
    elif a == 0 and p == 1:  fp += 1   # 실제 음성, 양성 예측 (오경보)
    elif a == 1 and p == 0:  fn += 1   # 실제 양성, 음성 예측 (위험!)

print('─' * 40)
print(f'✅ TP (진짜 양성):   {tp}개')
print(f'✅ TN (진짜 음성):   {tn}개')
print(f'⚠️  FP (오경보):      {fp}개')
print(f'🚨 FN (놓침):        {fn}개')
print('─' * 40)
print(f'합계: {tp+tn+fp+fn}개 (전체 데이터 수)')

In [ ]:
# ─── 방법 2: 한 줄 표현식으로 간단하게 ─────────────────────────────
tp2 = sum(a==1 and p==1 for a, p in zip(actual, pred))
tn2 = sum(a==0 and p==0 for a, p in zip(actual, pred))
fp2 = sum(a==0 and p==1 for a, p in zip(actual, pred))
fn2 = sum(a==1 and p==0 for a, p in zip(actual, pred))

print(f'한 줄 계산 결과: TP={tp2}, TN={tn2}, FP={fp2}, FN={fn2}')
print(f'for문 결과와 같나요? → {tp==tp2 and tn==tn2 and fp==fp2 and fn==fn2}')

In [ ]:
# ─── 혼동행렬 시각화 ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))

matrix = [[tp, fn],
          [fp, tn]]

colors_matrix = [['#4CAF82', '#E05C5C'],
                 ['#F9C642', '#5CB8B2']]

labels_matrix = [[f'TP\n{tp}', f'FN\n{fn}'],
                 [f'FP\n{fp}', f'TN\n{tn}']]

for i in range(2):
    for j in range(2):
        ax.add_patch(plt.Rectangle((j, 1-i), 1, 1,
                     facecolor=colors_matrix[i][j], alpha=0.85,
                     edgecolor='white', linewidth=3))
        ax.text(j+0.5, 1.5-i, labels_matrix[i][j],
                ha='center', va='center',
                fontsize=20, fontweight='bold', color='white')

ax.set_xlim(0, 2)
ax.set_ylim(0, 2)
ax.set_xticks([0.5, 1.5])
ax.set_xticklabels(['양성(1) 예측', '음성(0) 예측'], fontsize=12)
ax.set_yticks([0.5, 1.5])
ax.set_yticklabels(['실제 음성(0)', '실제 양성(1)'], fontsize=12)
ax.set_title('혼동행렬 (Confusion Matrix)', fontsize=14, fontweight='bold', pad=15)
ax.xaxis.tick_top()
ax.xaxis.set_label_position('top')
ax.set_xlabel('예측값 →', fontsize=12)

plt.tight_layout()
plt.show()

---
## 📈 STEP 4. 평가 지표 계산하기

| 지표 | 공식 | 의미 |
|------|------|------|
| **Accuracy** | (TP+TN) ÷ 전체 | 전체 중 맞춘 비율 |
| **Precision** | TP ÷ (TP+FP) | 양성 예측 중 진짜 양성 비율 |
| **Recall** | TP ÷ (TP+FN) | 실제 양성 중 맞게 찾은 비율 |
| **F1-Score** | 2×P×R ÷ (P+R) | Precision과 Recall의 균형 |

In [ ]:
# ─── 수식으로 직접 계산 ──────────────────────────────────────────
accuracy  = (tp + tn) / len(actual)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print('=' * 45)
print(f'📊 Accuracy  (정확도):  {accuracy:.2f}  = {accuracy*100:.0f}%')
print(f'🎯 Precision (정밀도):  {precision:.2f}  = {precision*100:.0f}%')
print(f'🔍 Recall    (재현율):  {recall:.2f}  = {recall*100:.0f}%')
print(f'⚖️  F1-Score  (균형):    {f1:.2f}  = {f1*100:.0f}%')
print('=' * 45)

print('\n💡 해석:')
print(f'  - 10개 중 {int(accuracy*10)}개를 정확히 예측했습니다.')
print(f'  - 양성이라고 예측한 것 중 {precision*100:.0f}%가 실제 양성입니다.')
print(f'  - 실제 양성을 {recall*100:.0f}% 놓치지 않고 찾아냈습니다.')

In [ ]:
# ─── 지표 시각화 ─────────────────────────────────────────────────
metrics_names  = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
metrics_values = [accuracy, precision, recall, f1]
bar_colors     = ['#4A90D9', '#5CB8B2', '#F07C3A', '#4CAF82']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(metrics_names, metrics_values,
              color=bar_colors, edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, metrics_values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', va='bottom',
            fontsize=14, fontweight='bold', color='#1A2B3C')

ax.set_ylim(0, 1.15)
ax.set_ylabel('점수', fontsize=12)
ax.set_title('평가 지표 비교', fontsize=14, fontweight='bold')
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='0.8 기준선')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

---
## ⚡ STEP 5. sklearn으로 한 번에 확인하기

수작업으로 계산한 값과 비교해 봅시다.

In [ ]:
# ─── sklearn 자동 계산 ────────────────────────────────────────────
sk_accuracy  = accuracy_score(actual, pred)
sk_precision = precision_score(actual, pred, zero_division=0)
sk_recall    = recall_score(actual, pred)
sk_f1        = f1_score(actual, pred)

print('─── sklearn 결과 ───────────────────────')
print(f'Accuracy:  {sk_accuracy:.2f}')
print(f'Precision: {sk_precision:.2f}')
print(f'Recall:    {sk_recall:.2f}')
print(f'F1-Score:  {sk_f1:.2f}')
print('───────────────────────────────────────')

# 수작업 결과와 동일한지 확인
same = (round(accuracy,2)==round(sk_accuracy,2) and
        round(precision,2)==round(sk_precision,2) and
        round(recall,2)==round(sk_recall,2) and
        round(f1,2)==round(sk_f1,2))
print(f'\n수작업 계산 결과와 동일? → {"✅ 네, 일치합니다!" if same else "❌ 다릅니다. 수식을 확인해보세요."}')

In [ ]:
# ─── 상세 분류 보고서 ────────────────────────────────────────────
print('📋 Classification Report')
print('=' * 55)
print(classification_report(actual, pred,
                             target_names=['불합격(0)', '합격(1)']))

# 혼동행렬 (sklearn)
cm = confusion_matrix(actual, pred)
print(f'혼동행렬 (sklearn):\n{cm}')
print('  형태: [[TN, FP], [FN, TP]]')

---
## 🛠️ STEP 6. 나만의 예측값으로 바꿔보기 (응용)

아래 `my_pred` 리스트를 직접 수정해서 결과가 어떻게 달라지는지 확인해보세요!

> **힌트**: 값은 0(음성/불합격) 또는 1(양성/합격)만 사용 가능합니다.

In [ ]:
# ══════════════════════════════════════════════════
# TODO: 아래 my_pred 값을 직접 바꿔보세요! 🎮
# ══════════════════════════════════════════════════
actual   = [1, 0, 1, 0, 1, 1, 0, 0, 1, 0]  # 실제값 (바꾸지 마세요)
my_pred  = [1, 0, 1, 0, 1, 1, 0, 0, 1, 0]  # ← 여기를 수정해보세요!
# ══════════════════════════════════════════════════

# 자동 계산
my_tp = sum(a==1 and p==1 for a,p in zip(actual, my_pred))
my_tn = sum(a==0 and p==0 for a,p in zip(actual, my_pred))
my_fp = sum(a==0 and p==1 for a,p in zip(actual, my_pred))
my_fn = sum(a==1 and p==0 for a,p in zip(actual, my_pred))

my_acc  = accuracy_score(actual, my_pred)
my_prec = precision_score(actual, my_pred, zero_division=0)
my_rec  = recall_score(actual, my_pred, zero_division=0)
my_f1   = f1_score(actual, my_pred, zero_division=0)

print(f'내 예측값: {my_pred}')
print(f'실제  값:  {actual}')
print()
print(f'TP={my_tp}  TN={my_tn}  FP={my_fp}  FN={my_fn}')
print(f'Accuracy={my_acc:.2f}  Precision={my_prec:.2f}  Recall={my_rec:.2f}  F1={my_f1:.2f}')

# 막대 차트
fig, ax = plt.subplots(figsize=(7, 3.5))
names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
vals  = [my_acc, my_prec, my_rec, my_f1]
ax.barh(names, vals, color=['#4A90D9','#5CB8B2','#F07C3A','#4CAF82'],
        edgecolor='white')
for i, v in enumerate(vals):
    ax.text(v+0.01, i, f'{v:.2f}', va='center', fontsize=12, fontweight='bold')
ax.set_xlim(0, 1.2)
ax.set_title('내 예측 결과 평가 지표', fontsize=13, fontweight='bold')
ax.axvline(x=0.8, color='gray', linestyle='--', alpha=0.5)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

---
## 🏆 STEP 7. 보너스 — 실제 머신러닝 모델 맛보기

Decision Tree를 사용해서 학생 합격 여부를 예측해봅시다.
(다음 주차에서 더 자세히 배웁니다!)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

# 데이터 준비
X = df[['공부시간(h)', '출석률(%)', '과제제출(개)']]
y = df['합격여부']

# 훈련/테스트 분리 (80:20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'훈련 데이터: {len(X_train)}개 (80%)')
print(f'테스트 데이터: {len(X_test)}개 (20%)')

# 모델 학습
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

print(f'\n실제값:  {list(y_test)}')
print(f'예측값:  {list(y_pred)}')
print(f'\n🎯 모델 Accuracy: {accuracy_score(y_test, y_pred):.2f}')
print()
print('📋 상세 결과:')
print(classification_report(y_test, y_pred,
                             target_names=['불합격', '합격'],
                             zero_division=0))

---
## ✅ 실습 완료!

### 오늘 배운 것 정리

| 개념 | 핵심 내용 |
|------|-----------|
| **Feature** | AI가 보는 입력 정보 (공부시간, 출석률 등) |
| **Label** | AI가 맞혀야 할 정답 (합격/불합격) |
| **TP/TN/FP/FN** | 예측 결과를 4가지로 분류 |
| **Accuracy** | 전체 정확도 = (TP+TN) ÷ 전체 |
| **Precision** | 양성 예측의 정확성 = TP ÷ (TP+FP) |
| **Recall** | 실제 양성을 놓치지 않는 정도 = TP ÷ (TP+FN) |
| **F1-Score** | Precision과 Recall의 균형 점수 |

### 🚀 다음 시간 예고
- **Decision Tree** 모델 직접 만들기
- **훈련/테스트 분리**의 실제 효과 확인
- **과적합(Overfitting)** 직접 실험해보기

---
*AI 기초 강의 | 3주차 실습*